In [ ]:
import pandas as pd
import requests
from pathlib import Path
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Path
CSV_PATH = "four_classes_image_index.csv"

OUTPUT_DIR = Path("dataset_4classes")
IMAGE_DIR = OUTPUT_DIR / "anh_tong"

OUTPUT_CSV = OUTPUT_DIR / "clean_dataset_8000.csv"

# 2. Cau hinh
CATEGORIES = [
    "Shirts",
    "Tshirts",
    "Casual Shoes",
    "Sports Shoes"
]

SAMPLES_PER_CATEGORY = 2000
RANDOM_STATE = 42
MAX_WORKERS = 8

data = pd.read_csv(CSV_PATH)

data["id"] = pd.to_numeric(
    data["id"], errors="coerce"
)

data = data.dropna(
    subset=["id", "link", "articleType"]
).copy()

data["id"] = data["id"].astype(int)

#Tao thu muc
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

#Ham tai anh
def download_image(row):
    image_id = int(row["id"])
    url = str(row["link"])

    try:
        response = requests.get(
            url,
            timeout=20,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        image = Image.open(
            BytesIO(response.content)
        )
        image.load()
        image = image.convert("RGB")

        save_path = IMAGE_DIR / f"{image_id}.jpg"
        image.save(save_path, format="JPEG")

        return row.to_dict()

    except Exception:
        return None


# Loc va tai 2000 anh moi nhan
all_results = []

for category in CATEGORIES:

    category_df = data[
        data["articleType"] == category
    ].copy()

    category_df = category_df.sample(
        frac=1,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    successful = []

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = [
            executor.submit(download_image, row)
            for _, row in category_df.iterrows()
        ]

        for future in as_completed(futures):
            result = future.result()

            if result is not None:
                successful.append(result)

                if len(successful) == SAMPLES_PER_CATEGORY:
                    for f in futures:
                        f.cancel()
                    break

 
    all_results.extend(successful)


#Luu CSV
final_df = pd.DataFrame(all_results)

final_df.to_csv(
    OUTPUT_CSV,
    index=False
)

Shirts: 2000/2000 anh
Tshirts: 2000/2000 anh
Casual Shoes: 2000/2000 anh
Sports Shoes: 2000/2000 anh
